# Lab 4 - Part 2: Document Classification, Sentiment Analysis & Topic Modeling

**Course:** Natural Language Processing

**Objectives:**
- Build document classifiers (intro + advanced)
- Perform sentiment analysis on different domains
- Discover topics using unsupervised learning
- Compare different feature extraction methods

---

## Instructions

1. Complete all exercises marked with `# YOUR CODE HERE`
2. **Answer all written questions** in the designated markdown cells
3. Save your completed notebook
4. **Push to your Git repository and send the link to: yoroba93@gmail.com**

### Personal Analysis Required

This lab contains questions requiring YOUR personal interpretation.

---

## Use Cases Covered

| Task | Intro Use Case | Advanced Use Case |
|------|----------------|-------------------|
| Classification | AG News | Legal Documents |
| Sentiment Analysis | Amazon Reviews | Twitter |
| Topic Modeling | Research Papers | Legal Contracts |

---

## Setup

In [ ]:
# Install required libraries (uncomment if needed)
!pip install datasets scikit-learn nltk pandas numpy matplotlib seaborn wordcloud gensim

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

# NLTK
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.pipeline import Pipeline

# Hugging Face datasets
from datasets import load_dataset

print("Setup complete!")

In [ ]:
# Common preprocessing function
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_simple(text):
    """Basic preprocessing: lowercase, remove punctuation."""
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return ' '.join(text.split())

def preprocess_advanced(text):
    """Advanced preprocessing: lowercase, remove punct, stopwords, lemmatize."""
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

print("Preprocessing functions ready!")

---

# PART A: Document Classification

We will work with two use cases:
1. **Intro:** News Topic Classification (AG News)
2. **Advanced:** Legal Document Classification (LexGLUE)

## A.1 Intro: News Topic Classification (AG News)

**Scenario:** A media company automatically routes articles to editorial teams.

**Feature Extraction:** TF-IDF

In [ ]:
!pip install -U datasets huggingface_hub

In [ ]:
# Load AG News dataset
print("Loading AG News dataset...")
ag_news = load_dataset("fancyzhx/ag_news")  # Fixed: added namespace

# Use subset for faster processing
ag_train = pd.DataFrame(ag_news['train']).sample(n=8000, random_state=42)
ag_test = pd.DataFrame(ag_news['test']).sample(n=2000, random_state=42)

# Label mapping
ag_labels = {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}
ag_train['label_name'] = ag_train['label'].map(ag_labels)
ag_test['label_name'] = ag_test['label'].map(ag_labels)

print(f"Train: {len(ag_train)}, Test: {len(ag_test)}")
print(f"\nCategories: {list(ag_labels.values())}")
print(ag_train['label_name'].value_counts())

In [ ]:
import re

# Define preprocessing function
def preprocess_simple(text):
    text = text.lower()                          # Lowercase
    text = re.sub(r'[^a-z0-9\s]', '', text)     # Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()     # Remove extra spaces
    return text

# Preprocess
ag_train['text_clean'] = ag_train['text'].apply(preprocess_simple)
ag_test['text_clean'] = ag_test['text'].apply(preprocess_simple)

# TF-IDF Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_ag = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), stop_words='english')

X_train_ag = tfidf_ag.fit_transform(ag_train['text_clean'])
X_test_ag = tfidf_ag.transform(ag_test['text_clean'])

y_train_ag = ag_train['label']
y_test_ag = ag_test['label']

print(f"TF-IDF features: {X_train_ag.shape[1]}")

In [ ]:
# Preprocess
ag_train['text_clean'] = ag_train['text'].apply(preprocess_simple)
ag_test['text_clean'] = ag_test['text'].apply(preprocess_simple)

# TF-IDF Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_ag = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), stop_words='english')

X_train_ag = tfidf_ag.fit_transform(ag_train['text_clean'])
X_test_ag = tfidf_ag.transform(ag_test['text_clean'])

y_train_ag = ag_train['label']
y_test_ag = ag_test['label']

print(f"TF-IDF features: {X_train_ag.shape[1]}")

### Exercise A.1: Train a News Classifier

In [ ]:
# TODO: Train a Logistic Regression classifier on AG News
# 1. Create the classifier
# 2. Train it
# 3. Make predictions
# 4. Calculate accuracy and F1-score (macro)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# Create LogisticRegression with increased max_iter to ensure convergence
clf_ag = LogisticRegression(max_iter=1000, random_state=42)

# Train the classifier on the TF-IDF training features
clf_ag.fit(X_train_ag, y_train_ag)

# Predict labels for the test set
y_pred_ag = clf_ag.predict(X_test_ag)

# Evaluate performance using accuracy and macro F1-score
accuracy_ag = accuracy_score(y_test_ag, y_pred_ag)
f1_ag = f1_score(y_test_ag, y_pred_ag, average='macro')

print(f"AG News Classification Results:")
print(f"  Accuracy: {accuracy_ag:.4f}")
print(f"  F1 (macro): {f1_ag:.4f}")

In [ ]:
from sklearn.metrics import classification_report

# Display classification report
print("\nClassification Report:")
print(classification_report(y_test_ag, y_pred_ag, target_names=list(ag_labels.values())))

## A.2 Advanced: Legal Document Classification (LexGLUE - ECtHR)

**Scenario:** A law firm classifies court decisions by violated articles.

**Feature Extraction:** Bag of Words with N-grams

**Challenge:** Legal text is longer and uses specialized vocabulary.

In [ ]:
# Load LexGLUE ECtHR dataset (European Court of Human Rights)
print("Loading LexGLUE ECtHR dataset...")
lex_glue = load_dataset("coastalcph/lex_glue", "ecthr_a")  # Fixed: added namespace

# Convert to DataFrame
lex_train = pd.DataFrame(lex_glue['train'])
lex_test = pd.DataFrame(lex_glue['test'])

# Use subset (legal docs are long)
lex_train = lex_train.sample(n=min(1500, len(lex_train)), random_state=42)
lex_test = lex_test.sample(n=min(500, len(lex_test)), random_state=42)

print(f"Train: {len(lex_train)}, Test: {len(lex_test)}")
print(f"\nColumns: {lex_train.columns.tolist()}")

In [ ]:
# Examine the data structure
print("Sample legal document (first 500 chars):")
sample_text = ' '.join(lex_train.iloc[0]['text'][:3])  # text is a list of paragraphs
print(sample_text[:500])

print(f"\nLabels (violated articles): {lex_train.iloc[0]['labels']}")

In [ ]:
# Prepare data: combine text paragraphs and use first label for simplicity
def prepare_legal_text(row):
    """Join text paragraphs and truncate."""
    full_text = ' '.join(row['text'])
    return full_text[:5000]  # Truncate long documents

lex_train['full_text'] = lex_train.apply(prepare_legal_text, axis=1)
lex_test['full_text'] = lex_test.apply(prepare_legal_text, axis=1)

# Use first label (multi-label to single-label for simplicity)
lex_train['primary_label'] = lex_train['labels'].apply(lambda x: x[0] if x else -1)
lex_test['primary_label'] = lex_test['labels'].apply(lambda x: x[0] if x else -1)

# Remove documents without labels
lex_train = lex_train[lex_train['primary_label'] >= 0]
lex_test = lex_test[lex_test['primary_label'] >= 0]

print(f"Cleaned - Train: {len(lex_train)}, Test: {len(lex_test)}")
print(f"\nLabel distribution:")
print(lex_train['primary_label'].value_counts().head(10))

### Exercise A.2: Build a Legal Document Classifier

In [ ]:
# TODO: Complete the legal document classifier using Bag of Words
from sklearn.feature_extraction.text import CountVectorizer

# Define advanced preprocessing function for legal documents
def preprocess_advanced(text):
    # Handle list input (LexGLUE stores text as list of paragraphs)
    if isinstance(text, list):
        text = ' '.join(text)
    text = text.lower()                            # Lowercase
    text = re.sub(r'\d+', 'NUM', text)            # Replace numbers with token
    text = re.sub(r'[^a-z\s]', '', text)          # Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()       # Remove extra spaces
    return text

# Step 1: Preprocess with advanced function
lex_train['text_clean'] = lex_train['full_text'].apply(preprocess_advanced)
lex_test['text_clean'] = lex_test['full_text'].apply(preprocess_advanced)

# Step 2: Create CountVectorizer (Bag of Words) with bigrams
# YOUR CODE HERE
# max_features=5000: keeps the 5000 most frequent tokens, balancing vocab size and performance
# ngram_range=(1,2): includes unigrams and bigrams to capture legal word pairs (e.g. "fair trial")
# min_df=2: ignores terms appearing in fewer than 2 documents (removes rare noise)
# max_df=0.95: ignores terms appearing in more than 95% of documents (removes stop-like words)
bow_legal = CountVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

# Step 3: Transform data
# fit_transform on train: learns vocabulary and encodes training documents
X_train_lex = bow_legal.fit_transform(lex_train['text_clean'])
# transform on test: encodes using vocabulary learned from train only (avoids data leakage)
X_test_lex = bow_legal.transform(lex_test['text_clean'])

y_train_lex = lex_train['primary_label']
y_test_lex = lex_test['primary_label']

print(f"BoW features: {X_train_lex.shape[1]}")

In [ ]:
# TODO: Train a Linear SVM classifier (good for high-dimensional legal text) or other model
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

# YOUR CODE HERE
# LinearSVC is well-suited for high-dimensional sparse data like BoW/TF-IDF vectors
clf_legal = LinearSVC(max_iter=2000, random_state=42)

# Train the classifier on the BoW training features
clf_legal.fit(X_train_lex, y_train_lex)

# Predict labels for the test set
y_pred_lex = clf_legal.predict(X_test_lex)

# Evaluate performance using accuracy and macro F1-score
accuracy_lex = accuracy_score(y_test_lex, y_pred_lex)
f1_lex = f1_score(y_test_lex, y_pred_lex, average='macro')

print(f"Legal Classification Results:")
print(f"  Accuracy: {accuracy_lex:.4f}")
print(f"  F1 (macro): {f1_lex:.4f}")

### Written Question A.1 (Personal Interpretation)

Compare your results from AG News and Legal classification:

1. **Which task achieved higher accuracy?** Why do you think there's a difference?
2. **What vectorizer parameters did you choose for legal text?** Justify each choice.
3. **What challenges are unique to legal document classification?** (Consider: length, vocabulary, ambiguity)

**YOUR ANSWER:**

1. Accuracy comparison:
   - AG News:  0.8890 | Legal: 0.6308
   - Reason for difference: AG News achieved higher accuracy than the legal classification task. This is expected because news articles follow relatively standard language patterns and are clearly categorized into distinct topics like Sports or Business. Legal documents, on the other hand, use highly specialized vocabulary and overlapping concepts across categories, making them harder to separate cleanly with traditional ML approaches.

2. My vectorizer choices:
   - max_features= 5000 because legal text has a large but repetitive vocabulary, so keeping the top 5000 terms captures the most meaningful legal terminology without overwhelming the model with noise.
   - ngram_range=(1,2) because legal language relies heavily on multi-word expressions like "fair trial" or "reasonable doubt", so bigrams help capture these meaningful pairs that unigrams alone would miss.
   - min_df=2 because very rare terms in legal documents are often case-specific proper nouns or citation numbers that don't generalize well across documents.
   - max_df=0.95 because terms appearing in almost every document, like "shall" or "party", behave like stopwords in legal text and add little discriminative value.

3. Legal classification challenges:
   - Legal documents are significantly longer than news articles, which means important classification signals can be buried deep in the text and truncation can cause information loss. The vocabulary is highly specialized and often ambiguous, where the same term can mean different things depending on legal context. Finally, legal categories frequently overlap, for example a contract can involve employment, confidentiality, and IP clauses all at once, making single-label classification inherently imprecise.

---

# PART B: Sentiment Analysis

We will work with two use cases:
1. **Intro:** E-commerce Product Reviews (Amazon)
2. **Advanced:** Social Media Sentiment (Twitter/TweetEval)

## B.1 Intro: Amazon Product Reviews

**Scenario:** An e-commerce company monitors product sentiment.

**Feature Extraction:** TF-IDF

In [ ]:

# NOTE: Originally used 'amazon_reviews_multi' (1-5 stars, 5 classes) but it's no longer
# supported in recent versions of the datasets library (RuntimeError: Dataset scripts are
# no longer supported). Replaced with 'fancyzhx/amazon_polarity' which is binary
# (negative=0, positive=1), mapped to stars 1 and 5. Same learning objective, simpler task.

# Load Amazon Polarity dataset (binary sentiment: negative=0, positive=1)
print("Loading Amazon Reviews dataset...")
amazon = load_dataset("fancyzhx/amazon_polarity")  # Replaces broken amazon_reviews_multi

# Convert to DataFrame and sample
amazon_train = pd.DataFrame(amazon['train']).sample(n=5000, random_state=42)
amazon_test = pd.DataFrame(amazon['test']).sample(n=1000, random_state=42)

# Rename columns to match the rest of your notebook
# 'content' = review text, 'label' = 0 (negative) or 1 (positive)
amazon_train['stars'] = amazon_train['label'].map({0: 1, 1: 5})  # Map to star-like scale
amazon_test['stars'] = amazon_test['label'].map({0: 1, 1: 5})
amazon_train['review_body'] = amazon_train['content']
amazon_test['review_body'] = amazon_test['content']

print(f"Train: {len(amazon_train)}, Test: {len(amazon_test)}")
print(f"\nColumns: {amazon_train.columns.tolist()}")
print(f"\nStar rating distribution:")
print(amazon_train['stars'].value_counts().sort_index())

In [ ]:
# Convert to binary sentiment (1-2 stars = negative, 4-5 stars = positive)
# Remove neutral (3 stars) for clearer distinction

def to_binary_sentiment(stars):
    if stars <= 2:
        return 0  # Negative
    elif stars >= 4:
        return 1  # Positive
    else:
        return -1  # Neutral (to be removed)

amazon_train['sentiment'] = amazon_train['stars'].apply(to_binary_sentiment)
amazon_test['sentiment'] = amazon_test['stars'].apply(to_binary_sentiment)

# Remove neutral
amazon_train = amazon_train[amazon_train['sentiment'] >= 0]
amazon_test = amazon_test[amazon_test['sentiment'] >= 0]

sentiment_labels = {0: 'Negative', 1: 'Positive'}
print(f"After filtering - Train: {len(amazon_train)}, Test: {len(amazon_test)}")
print(f"\nSentiment distribution:")
print(amazon_train['sentiment'].value_counts())

In [ ]:
# Note: amazon_polarity has no 'product_category' or 'sentiment' columns;
# using 'title' and 'label' instead
amazon_train['sentiment'] = amazon_train['label']  # 0 = negative, 1 = positive
amazon_test['sentiment'] = amazon_test['label']

# Show sample reviews
print("Sample POSITIVE review:")
pos_sample = amazon_train[amazon_train['sentiment'] == 1].iloc[0]
print(f"Title: {pos_sample['title']}")
print(f"Review: {pos_sample['review_body'][:300]}...")

print("\n" + "="*60 + "\n")

print("Sample NEGATIVE review:")
neg_sample = amazon_train[amazon_train['sentiment'] == 0].iloc[0]
print(f"Title: {neg_sample['title']}")
print(f"Review: {neg_sample['review_body'][:300]}...")

In [ ]:
# Show sample reviews
# Note: amazon_polarity has no 'product_category'; using 'title' instead
print("Sample POSITIVE review:")
pos_sample = amazon_train[amazon_train['sentiment'] == 1].iloc[0]
print(f"Title: {pos_sample['title']}")
print(f"Review: {pos_sample['review_body'][:300]}...")

print("\n" + "="*60 + "\n")

print("Sample NEGATIVE review:")
neg_sample = amazon_train[amazon_train['sentiment'] == 0].iloc[0]
print(f"Title: {neg_sample['title']}")
print(f"Review: {neg_sample['review_body'][:300]}...")

### Exercise B.1: Build Amazon Sentiment Classifier

In [ ]:
from sklearn.naive_bayes import MultinomialNB

In [ ]:
# TODO: Build sentiment classifier for Amazon reviews

# Step 1: Preprocess
amazon_train['text_clean'] = amazon_train['review_body'].apply(preprocess_simple)
amazon_test['text_clean'] = amazon_test['review_body'].apply(preprocess_simple)

# Step 2: TF-IDF
tfidf_amazon = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

X_train_amz = tfidf_amazon.fit_transform(amazon_train['text_clean'])
X_test_amz = tfidf_amazon.transform(amazon_test['text_clean'])
y_train_amz = amazon_train['sentiment']
y_test_amz = amazon_test['sentiment']

# Step 3: Train Naive Bayes
clf_amazon = MultinomialNB(alpha=1.0)
clf_amazon.fit(X_train_amz, y_train_amz)

# Step 4: Predict
y_pred_amz = clf_amazon.predict(X_test_amz)

# Evaluate
print(f"Amazon Sentiment Results:")
print(f"  Accuracy: {accuracy_score(y_test_amz, y_pred_amz):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test_amz, y_pred_amz, target_names=['Negative', 'Positive']))

In [ ]:
# Analyze most predictive words
feature_names = tfidf_amazon.get_feature_names_out()

# For Naive Bayes, use log probabilities
neg_probs = clf_amazon.feature_log_prob_[0]
pos_probs = clf_amazon.feature_log_prob_[1]
log_ratio = pos_probs - neg_probs

# Top positive and negative words
top_pos_idx = log_ratio.argsort()[-15:]
top_neg_idx = log_ratio.argsort()[:15]

print("Top POSITIVE words:", [feature_names[i] for i in top_pos_idx])
print("\nTop NEGATIVE words:", [feature_names[i] for i in top_neg_idx])

## B.2 Advanced: Twitter Sentiment (TweetEval)

**Scenario:** A brand monitors social media sentiment about their products.

**Feature Extraction:** Bag of Words with character n-grams (better for informal text)

**Challenge:** Tweets are short, informal, with hashtags, mentions, and slang.

In [ ]:
# Load TweetEval sentiment dataset
print("Loading TweetEval Sentiment dataset...")
tweet_eval = load_dataset("cardiffnlp/tweet_eval", "sentiment")

tweet_train = pd.DataFrame(tweet_eval['train'])
tweet_test = pd.DataFrame(tweet_eval['test'])

# Labels: 0=negative, 1=neutral, 2=positive
tweet_labels = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
tweet_train['label_name'] = tweet_train['label'].map(tweet_labels)
tweet_test['label_name'] = tweet_test['label'].map(tweet_labels)

print(f"Train: {len(tweet_train)}, Test: {len(tweet_test)}")
print(f"\nLabel distribution:")
print(tweet_train['label_name'].value_counts())

In [ ]:
# Sample tweets
for label in [0, 1, 2]:
    sample = tweet_train[tweet_train['label'] == label].iloc[0]
    print(f"[{tweet_labels[label]}]: {sample['text']}\n")

In [ ]:
# Special preprocessing for tweets
def preprocess_tweet(text):
    """Preprocess tweet text."""
    text = str(text).lower()
    # Keep @mentions and #hashtags but simplify
    text = re.sub(r'@\w+', '@user', text)  # Replace mentions with @user
    text = re.sub(r'http\S+', 'URL', text)  # Replace URLs
    text = re.sub(r'[^a-zA-Z@#\s]', '', text)  # Keep @ and # symbols
    return ' '.join(text.split())

tweet_train['text_clean'] = tweet_train['text'].apply(preprocess_tweet)
tweet_test['text_clean'] = tweet_test['text'].apply(preprocess_tweet)

print("Sample preprocessed tweet:")
print(f"Original: {tweet_train.iloc[0]['text']}")
print(f"Cleaned:  {tweet_train.iloc[0]['text_clean']}")

### Exercise B.2: Build Twitter Sentiment Classifier

In [ ]:
# TODO: Build a classifier using character n-grams (good for short, informal text)

# Create a vectorizer with character n-grams
# 'char_wb' es ideal para texto informal como tweets: respeta los límites de palabra
char_vectorizer = TfidfVectorizer(
    analyzer='char_wb',     # 'char_wb' for character n-grams with word boundaries
    ngram_range=(3, 6),     # Captura secuencias de 3 a 6 caracteres (óptimo para tweets)
    max_features=5000,      # Limitamos el vocabulario a las 5000 features más frecuentes
    min_df=2                # Ignoramos n-gramas que aparecen en menos de 2 documentos
)

# Ajustamos el vectorizador con el train y transformamos ambos splits
# IMPORTANTE: solo fit_transform en train, transform en test (evita data leakage)
X_train_tw = char_vectorizer.fit_transform(tweet_train['text'])
X_test_tw = char_vectorizer.transform(tweet_test['text'])

y_train_tw = tweet_train['label']
y_test_tw = tweet_test['label']

print(f"Character n-gram features: {X_train_tw.shape[1]}")

In [ ]:

# TODO: Train Logistic Regression and evaluate

# Logistic Regression works well with TF-IDF and multiclass problems
clf_tweet = LogisticRegression(
    max_iter=1000,      # Enough iterations to converge
    C=1.0,              # Standard regularization strength
    solver='lbfgs',     # Efficient solver for multiclass problems
    multi_class='auto'  # Automatically detects multiclass (3 classes)
)

# Train the model and generate predictions
clf_tweet.fit(X_train_tw, y_train_tw)
y_pred_tw = clf_tweet.predict(X_test_tw)

# Evaluate
print(f"Twitter Sentiment Results (3-class):")
print(f"  Accuracy: {accuracy_score(y_test_tw, y_pred_tw):.4f}")
print(f"  F1 (macro): {f1_score(y_test_tw, y_pred_tw, average='macro'):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test_tw, y_pred_tw, target_names=list(tweet_labels.values())))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Confusion matrix
cm_tw = confusion_matrix(y_test_tw, y_pred_tw)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_tw, annot=True, fmt='d', cmap='Blues',
            xticklabels=list(tweet_labels.values()),
            yticklabels=list(tweet_labels.values()))
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Twitter Sentiment Confusion Matrix')
plt.tight_layout()
plt.savefig('twitter_sentiment_cm.png', dpi=150)
plt.show()

### Written Question B.1 (Personal Interpretation)

Compare Amazon vs Twitter sentiment analysis:

1. **Which task was harder?** Look at the F1 scores and confusion matrices.
2. **Why did you choose those character n-gram parameters for Twitter?** What's the advantage over word n-grams?
3. **Looking at the Twitter confusion matrix, which class is most often confused?** Why might this be?
4. **Give an example tweet that would be hard to classify correctly.** Explain why.

**YOUR ANSWER:**

1. Harder task:
   - Amazon F1: 0.8570 | Twitter F1: 0.5575
   - Reason: Twitter sentiment was harder. Amazon reviews tend to be explicit and detailed, making positive and negative signals easy to detect. Twitter is a 3-class problem (negative, neutral, positive) with very short and informal text, which naturally lowers F1 scores. The neutral class especially hurts performance since many tweets sit on the boundary between sentiment categories.

2. Character n-gram choices:
   - ngram_range= (3,6) because character sequences of that length capture enough context to recognize word fragments, suffixes, and informal spellings common in tweets like "looove" or "omg" without needing the full word to be correctly spelled or tokenized.
   - Advantage over words: The advantage over word n-grams is that character n-grams are robust to typos, slang, hashtags, and abbreviations that are extremely common on Twitter. A word-level model would treat "amazing", "amaazing", and "amazinggg" as completely different tokens, while character n-grams naturally capture their similarity.

3. Most confused class:
   - Class: Neutral
   - Reason: This happens because neutral tweets often contain sentiment words used in a non-opinionated way, for example reporting a fact like "The game ended 3-0" which contains no real sentiment but shares vocabulary with both positive and negative tweets. The boundary between mild positivity and neutrality is especially blurry.

4. Difficult tweet example:
   - Tweet: "Well that was certainly something..."
   - Why it's hard: This is hard to classify because the sarcasm and vagueness make it impossible to determine sentiment without broader context. It could be mildly negative, neutral, or even positive depending on what the person is referring to. A bag-of-words or character n-gram model has no way to detect implied tone or sarcasm from surface text alone.

---

# PART C: Topic Modeling

We will work with two use cases:
1. **Intro:** Research Paper Topics (ArXiv)
2. **Advanced:** Legal Contract Topics

## C.1 Intro: Research Paper Topic Discovery (ArXiv)

**Scenario:** A research organization discovers themes in scientific papers.

**Method:** LDA (Latent Dirichlet Allocation)

In [ ]:
import itertools

In [ ]:
# Load ArXiv papers dataset
print("Loading ArXiv papers dataset...")
arxiv = load_dataset("ccdv/arxiv-classification", split="train")

# Sample 2000 papers from the dataset
arxiv_df = pd.DataFrame(arxiv).sample(n=2000, random_state=42)

print(f"Loaded {len(arxiv_df)} papers")
print(f"Columns: {arxiv_df.columns.tolist()}")

In [ ]:
print(arxiv_df.columns.tolist())
print(arxiv_df.iloc[0])

In [ ]:
# Examine sample
print("Sample paper abstract (first 500 chars):")
print(arxiv_df.iloc[0]['text'][:500])

In [ ]:
# Preprocess abstracts for topic modeling
arxiv_df['abstract_clean'] = arxiv_df['text'].apply(preprocess_advanced)

# Create document-term matrix with CountVectorizer
count_vec_arxiv = CountVectorizer(
    max_features=5000,  # Limit vocabulary to top 5000 words
    min_df=2,           # Ignore terms appearing in less than 2 documents
    max_df=0.95         # Ignore terms appearing in more than 95% of documents
)

dtm_arxiv = count_vec_arxiv.fit_transform(arxiv_df['abstract_clean'])
print(f"Document-term matrix: {dtm_arxiv.shape}")

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

# Train LDA model
n_topics_arxiv = 10  # 10 topics works well for diverse scientific papers

lda_arxiv = LatentDirichletAllocation(
    n_components=n_topics_arxiv,
    random_state=42,
    max_iter=15,
    learning_method='online'
)

print("Training LDA on ArXiv papers...")
lda_arxiv.fit(dtm_arxiv)
print("Done!")

In [ ]:
# Display topics
def display_lda_topics(model, feature_names, n_words=12):
    """Display top words for each LDA topic."""
    for topic_idx, topic in enumerate(model.components_):
        top_words_idx = topic.argsort()[:-n_words-1:-1]
        top_words = [feature_names[i] for i in top_words_idx]
        print(f"Topic {topic_idx}: {', '.join(top_words)}")

feature_names_arxiv = count_vec_arxiv.get_feature_names_out()
print("ArXiv Paper Topics (LDA):")
print("=" * 70)
display_lda_topics(lda_arxiv, feature_names_arxiv)

### Exercise C.1: Interpret ArXiv Topics

In [ ]:
# Show top words per topic
feature_names = count_vec_arxiv.get_feature_names_out()
for topic_idx, topic in enumerate(lda_arxiv.components_):
    top_words = [feature_names[i] for i in topic.argsort()[:-11:-1]]
    print(f"Topic {topic_idx}: {', '.join(top_words)}")

In [ ]:
# TODO: Assign meaningful labels to each topic based on the keywords
my_arxiv_topic_labels = {
    0: "Linear Algebra & Optimization",       # matrix, linear, vector, matrices
    1: "Coding Theory & Metric Spaces",        # codes, distance, metric, boundary
    2: "Multilingual / Game Theory",           # de, la, et, game (mixed languages + game)
    3: "Deep Learning & Computer Vision",      # learning, model, training, image, neural
    4: "Abstract Algebra & Formal Proofs",     # theorem, group, proof, lemma, finite
    5: "Logic & Type Theory",                  # type, definition, variables, rule, proof
    6: "Programming Languages & Software",     # program, code, programming, language
    7: "Graph Algorithms & Complexity",        # graph, edge, vertex, algorithm, time
    8: "Control Systems & Signal Processing",  # system, control, power, fig
    9: "Probability & Statistics",             # probability, distribution, theorem, random
}

print("My Topic Interpretations:")
for topic_id, label in my_arxiv_topic_labels.items():
    if label != "___":
        print(f"  Topic {topic_id}: {label}")

## C.2 Advanced: Legal Contract Topic Discovery

**Scenario:** A law firm discovers themes across contracts to organize their database.

**Method:** NMF (Non-negative Matrix Factorization) - often better for shorter, specialized documents

**Challenge:** Legal language is formal and domain-specific.

In [ ]:
# Note: 'albertvillanova/legal_contracts' is no longer supported (uses a legacy loading script).
# Using 'pile-of-law/pile-of-law' with the 'atticus_contracts' subset as a replacement,
# which contains the same type of full legal contract texts in a modern Parquet format.

In [ ]:
# Load legal contracts dataset (streaming to handle large size)
print("Loading Legal Contracts dataset...")
legal_stream = load_dataset("hugsid/legal-contracts", split="train", streaming=True)

# Take first 1500 contracts
legal_contracts = []
for i, item in enumerate(legal_stream):
    if i >= 1500:
        break
    legal_contracts.append(item)

legal_df = pd.DataFrame(legal_contracts)
print(f"Loaded {len(legal_df)} contracts")
print(f"Columns: {legal_df.columns.tolist()}")

In [ ]:
# Preprocess legal text (truncate long documents)
legal_df['text_truncated'] = legal_df['text'].str[:8000]  # Truncate
legal_df['text_clean'] = legal_df['text_truncated'].apply(preprocess_advanced)

print("Sample contract (cleaned, first 300 chars):")
print(legal_df.iloc[0]['text_clean'][:300])

### Exercise C.2: Build NMF Topic Model for Legal Contracts

In [ ]:
# TODO: Create TF-IDF vectorizer for NMF (NMF works better with TF-IDF)

# TF-IDF is preferred over CountVectorizer for NMF as it down-weights common terms
tfidf_legal = TfidfVectorizer(
    max_features=5000,  # Limit vocabulary to top 5000 words
    min_df=2,           # Ignore terms appearing in less than 2 documents
    max_df=0.95         # Ignore terms appearing in more than 95% of documents
)

dtm_legal = tfidf_legal.fit_transform(legal_df['text_clean'])
print(f"Legal document-term matrix: {dtm_legal.shape}")

In [ ]:
from sklearn.decomposition import NMF

# TODO: Train NMF model
# Choose number of topics (legal contracts may have: employment, confidentiality, IP, services, etc.)

n_topics_legal = 8  # 8 topics covers common legal contract types well

nmf_legal = NMF(
    n_components=n_topics_legal,
    random_state=42,
    max_iter=200
)

print(f"Training NMF with {n_topics_legal} topics...")
nmf_legal.fit(dtm_legal)
print("Done!")

In [ ]:
# Display NMF topics
def display_nmf_topics(model, feature_names, n_words=12):
    """Display top words for each NMF topic."""
    for topic_idx, topic in enumerate(model.components_):
        top_words_idx = topic.argsort()[:-n_words-1:-1]
        top_words = [feature_names[i] for i in top_words_idx]
        print(f"Topic {topic_idx}: {', '.join(top_words)}")

feature_names_legal = tfidf_legal.get_feature_names_out()
print(f"Legal Contract Topics (NMF, {n_topics_legal} topics):")
print("=" * 70)
display_nmf_topics(nmf_legal, feature_names_legal)

In [ ]:
# TODO: Assign labels to legal topics

my_legal_topic_labels = {}  # Add your labels: {0: "label", 1: "label", ...}

# Fill the dictionary based on top keywords per topic
my_legal_topic_labels = {
    0: "Confidentiality & Affiliates",       # affiliates, confidential, information, negotiations
    1: "Transaction & Disclosure Parties",   # party, disclosing, transaction, prior
    2: "Contract Clauses & Dates",           # clause, date, upon, into
    3: "Restrictions & Prior Conditions",    # shall, prior, without, save
    4: "Confidential Information Handling",  # confidential, analyses, relating, form
    5: "Negotiation Status & Facts",         # fact, negotiations, status, concerning
    6: "Written Materials & Disclosure",     # written, materials, confidential, upon
    7: "Representatives & Information Use",  # representatives, materials, whether, by
}

print("My Legal Topic Interpretations:")
for topic_id, label in my_legal_topic_labels.items():
    if label != "___":
        print(f"  Topic {topic_id}: {label}")

### Exercise C.3: Topic Distribution Visualization

In [ ]:
# Get document-topic distributions
doc_topics_legal = nmf_legal.transform(dtm_legal)

# Assign dominant topic
legal_df['dominant_topic'] = doc_topics_legal.argmax(axis=1)

# Visualize topic distribution
plt.figure(figsize=(10, 6))
topic_counts = legal_df['dominant_topic'].value_counts().sort_index()
bars = plt.bar(topic_counts.index, topic_counts.values, color=plt.cm.Set3(range(len(topic_counts))))
plt.xlabel('Topic')
plt.ylabel('Number of Contracts')
plt.title('Distribution of Contracts Across Topics')
plt.xticks(range(n_topics_legal))

# Add count labels
for bar, count in zip(bars, topic_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             str(count), ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('legal_topic_distribution.png', dpi=150)
plt.show()

In [ ]:
print(topic_counts)

### Written Question C.1 (Personal Interpretation)

Compare ArXiv (LDA) vs Legal Contracts (NMF) topic modeling:

1. **Which set of topics was easier to interpret?** Why?
2. **Looking at the legal topic distribution, is it balanced?** What does this tell you about the contract dataset?
3. **For each domain, if applicable, suggest 2 topics that might be merged and 1 topic that should be split.** Justify.

**YOUR ANSWER:**

1. Easier to interpret:
   - Domain: ArXiv
   - Reason: The topics are more distinct and specialized. Topics such as Deep Learning, Graph Algorithms, Probability, and Linear Algebra have clear keywords and are easier to label.

2. Legal topic distribution:
   - Balanced? No. Some topics contain many more documents than others. For example, Topic 1 has 419 documents while Topic 5 has only 48.
   - What this indicates: The contract dataset is not evenly distributed across legal themes. Some contract clauses appear much more frequently than others, suggesting that certain legal concepts are more common in the documents.

3. Topic refinement suggestions:
   - ArXiv - Merge: Topics 0 and 7 because both contain algorithmic and computational concepts with overlapping terminology.
   - ArXiv - Split: Topic 2 because it combines different themes and contains less coherent keywords.
   - Legal - Merge: Topics 4 and 6 because both focus on confidentiality and information disclosure concepts.
   - Legal - Split: Topic 1 because it is broad and may contain multiple legal themes that could be separated into more specific topics.

---

## Summary - Lab 4 Part 2

### Methods Summary

| Task | Dataset | Feature Extraction | Model |
|------|---------|-------------------|-------|
| Classification (Intro) | AG News | TF-IDF | Logistic Regression |
| Classification (Advanced) | LexGLUE | Bag of Words | Linear SVM |
| Sentiment (Intro) | Amazon Reviews | TF-IDF | Naive Bayes |
| Sentiment (Advanced) | Twitter | Character N-grams | Logistic Regression |
| Topic Modeling (Intro) | ArXiv | Count Vectors | LDA |
| Topic Modeling (Advanced) | Legal Contracts | TF-IDF | NMF |

### Key Takeaways

- **Classification:** TF-IDF works well for standard text; specialized domains need careful preprocessing
- **Sentiment:** Character n-grams help with informal/noisy text like tweets
- **Topic Modeling:** LDA assumes documents have multiple topics; NMF often gives cleaner topics for specialized domains

---

## Submission Checklist

- [ ] All code exercises completed (fill all `___` placeholders)
- [ ] **All written questions answered with YOUR personal interpretation**
- [ ] All visualizations saved (PNG files)
- [ ] Notebook saved
- [ ] Pushed to Git repository
- [ ] **Repository link sent to: yoroba93@gmail.com**
